# EXZECO Import Validation and Fix

This notebook demonstrates how to resolve the ImportError issues with relative imports in the EXZECO flood risk assessment project. The error occurs when trying to import modules that use relative imports (`from .core import`) while running in a Jupyter notebook environment.

## Problem Statement

The error message:
```
ImportError: attempted relative import with no known parent package
```

This happens because Python requires packages to be properly recognized when using relative imports, and direct script execution or notebook execution doesn't establish the proper package context.

## 1. Understanding the ImportError

Let's first examine the current state and understand why the ImportError occurs.

In [ ]:
# Let's try the problematic import to see the exact error
import sys
import os

print("Python executable:", sys.executable)
print("Current working directory:", os.getcwd())
print("Python path entries:")
for i, path in enumerate(sys.path):
    print(f"  {i}: {path}")

# Try the import that's failing
try:
    from exzeco import ExzecoAnalysis, ExzecoConfig, load_config
    print("✅ Import successful!")
except ImportError as e:
    print(f"❌ ImportError: {e}")
    print("\nThis confirms we have the relative import issue.")

## 2. Check Current Working Directory and Project Structure

Let's examine the project structure to understand the module hierarchy.

In [ ]:
# Check project structure
from pathlib import Path

# Current working directory
cwd = Path.cwd()
print(f"Current working directory: {cwd}")

# Check if we're in the project root
project_files = ['src', 'notebooks', 'config', 'requirements.txt']
print(f"\nProject root structure:")
for item in project_files:
    path = cwd / item
    exists = "✅" if path.exists() else "❌"
    print(f"  {exists} {item}")

# Check src directory structure
src_dir = cwd / 'src'
if src_dir.exists():
    print(f"\nSrc directory contents:")
    for item in src_dir.iterdir():
        if item.is_dir():
            print(f"  📁 {item.name}/")
            # Check for __init__.py in subdirectories
            init_file = item / '__init__.py'
            init_exists = "✅" if init_file.exists() else "❌"
            print(f"     {init_exists} __init__.py")
        else:
            print(f"  📄 {item.name}")

# Check core directory structure
core_dir = src_dir / 'core'
if core_dir.exists():
    print(f"\nCore module contents:")
    for item in core_dir.iterdir():
        print(f"  📄 {item.name}")

## 3. Add Project Root to Python Path

Now let's fix the import issue by adding the src directory to the Python path.

In [ ]:
# Add src directory to Python path
import sys
from pathlib import Path

# Get the src directory path
src_dir = Path.cwd() / 'src'
src_dir_str = str(src_dir.resolve())

print(f"Adding to Python path: {src_dir_str}")

# Add src directory to Python path if not already there
if src_dir_str not in sys.path:
    sys.path.append(src_dir_str)
    print("✅ Added src directory to Python path")
else:
    print("ℹ️  Src directory already in Python path")

# Show updated Python path
print(f"\nUpdated Python path:")
for i, path in enumerate(sys.path):
    marker = " 🎯" if path == src_dir_str else ""
    print(f"  {i}: {path}{marker}")

## 4. Verify the Import Fixes are Working

The import should now work because I've already updated the source files to handle relative import issues with fallback mechanisms.

In [ ]:
# Test the fixed imports
print("🧪 Testing EXZECO imports...")

try:
    # Test core module imports first
    print("\n1. Testing core module imports:")
    from core.flow_analysis import FlowAnalyzer
    print("   ✅ FlowAnalyzer imported successfully")
    
    from core.monte_carlo import MonteCarloSimulator
    print("   ✅ MonteCarloSimulator imported successfully")
    
    from core.geometry_processing import GeometryProcessor
    print("   ✅ GeometryProcessor imported successfully")
    
    from core.drainage_classification import DrainageClassifier, ClassificationThresholds
    print("   ✅ DrainageClassifier imported successfully")
    
    from core.export_utils import ResultExporter
    print("   ✅ ResultExporter imported successfully")
    
except ImportError as e:
    print(f"   ❌ Core module import failed: {e}")

print("\n2. Testing main EXZECO imports:")
try:
    from exzeco import ExzecoAnalysis, ExzecoConfig, load_config
    print("   ✅ ExzecoAnalysis, ExzecoConfig, load_config imported successfully")
    
    from dem_utils import DEMDownloader, StudyArea
    print("   ✅ DEMDownloader, StudyArea imported successfully")
    
    from visualization import ExzecoVisualizer
    print("   ✅ ExzecoVisualizer imported successfully")
    
    print("\n🎉 All imports successful! The relative import issue has been resolved.")
    
except ImportError as e:
    print(f"   ❌ Main import failed: {e}")
    print("\nLet's debug this further...")

## 5. Alternative Method: Using sys.path.insert()

If the append method doesn't work, we can try inserting the path at the beginning of the search order.

In [ ]:
# Alternative: Use sys.path.insert() for higher priority
import sys
from pathlib import Path

# Get the src directory path
src_dir = Path.cwd() / 'src'
src_dir_str = str(src_dir.resolve())

# Insert at the beginning of the path for higher priority
if src_dir_str not in sys.path:
    sys.path.insert(0, src_dir_str)
    print(f"✅ Inserted src directory at beginning of Python path: {src_dir_str}")
else:
    print(f"ℹ️  Src directory already in Python path: {src_dir_str}")

# Also ensure importlib is available for reloading modules if needed
import importlib

print("\n🔄 Now modules can be imported with proper priority and reloaded if needed.")

## 6. Verify Module Structure and __init__.py Files

Let's verify that all the package structure is correct.

In [ ]:
# Check module structure and __init__.py files
from pathlib import Path

def check_package_structure(base_path, package_name=""):
    """Check if directory structure is suitable for Python packages"""
    base = Path(base_path)
    
    if not base.exists():
        print(f"❌ Directory doesn't exist: {base}")
        return False
    
    # Check for __init__.py
    init_file = base / '__init__.py'
    init_exists = init_file.exists()
    init_status = "✅" if init_exists else "❌"
    
    prefix = f"{package_name}/" if package_name else ""
    print(f"{init_status} {prefix}__init__.py")
    
    # Check subdirectories
    for item in base.iterdir():
        if item.is_dir() and not item.name.startswith('.') and not item.name == '__pycache__':
            check_package_structure(item, f"{prefix}{item.name}")
    
    return init_exists

print("📦 Checking package structure:")
print("=" * 40)

# Check src directory
src_dir = Path.cwd() / 'src'
check_package_structure(src_dir, "src")

print("\n📋 Summary:")
print("- All directories that should be Python packages need __init__.py files")
print("- The fixes I applied should handle import issues gracefully")
print("- Both relative and absolute import approaches are now supported")

## 7. Final Test: Complete Import Validation

Let's do a comprehensive test of all the fixed imports to ensure everything works correctly.

In [ ]:
# Final comprehensive test
print("🔬 COMPREHENSIVE IMPORT VALIDATION")
print("=" * 50)

import_tests = [
    ("Core Flow Analysis", "core.flow_analysis", "FlowAnalyzer"),
    ("Core Monte Carlo", "core.monte_carlo", "MonteCarloSimulator"), 
    ("Core Geometry", "core.geometry_processing", "GeometryProcessor"),
    ("Core Drainage", "core.drainage_classification", "DrainageClassifier"),
    ("Core Export", "core.export_utils", "ResultExporter"),
    ("Main EXZECO", "exzeco", "ExzecoAnalysis"),
    ("DEM Utils", "dem_utils", "DEMDownloader"),
    ("Visualization", "visualization", "ExzecoVisualizer"),
]

success_count = 0
total_tests = len(import_tests)

for test_name, module_name, class_name in import_tests:
    try:
        # Dynamic import
        module = __import__(module_name, fromlist=[class_name])
        cls = getattr(module, class_name)
        print(f"✅ {test_name}: {module_name}.{class_name}")
        success_count += 1
    except ImportError as e:
        print(f"❌ {test_name}: Import failed - {e}")
    except AttributeError as e:
        print(f"⚠️  {test_name}: Module imported but class not found - {e}")
    except Exception as e:
        print(f"🔥 {test_name}: Unexpected error - {e}")

print(f"\n📊 RESULTS: {success_count}/{total_tests} imports successful")

if success_count == total_tests:
    print("🎉 ALL IMPORTS WORKING! The relative import issues have been resolved.")
    print("\n🚀 You can now run the main EXZECO notebook without ImportError issues.")
else:
    print("⚠️  Some imports still failing. Check the error messages above for details.")

print("\n💡 SOLUTION SUMMARY:")
print("1. ✅ Fixed relative imports in exzeco.py with try/except fallback")
print("2. ✅ Fixed relative imports in core/__init__.py with fallback")
print("3. ✅ Fixed relative imports in core/monte_carlo.py")
print("4. ✅ Added robust import handling in dem_utils.py")
print("5. ✅ Added sys.path management for notebook execution")